# 🧬 07 — Expert v2 "População": dados REAIS + pool de oponentes + gate devastador

> **Célula única — `Runtime → Run all`.** Secrets/configuração: `GH_TOKEN`, `HF_TOKEN`, `HF_REPO_ID`, `PROJECT_COMMIT`, `HF_BASELINE_REVISION`, identidade/fusos PHH, os campos `POKER_PHH_GOVERNANCE_*`/`POKER_PHH_*_REFERENCE` e os quatro IDs/referências `POKER_CANDIDATE_*`. Comece em CPU; use L4 apenas após profiling mostrar batching/utilização adequados.

## Por que este notebook existe
A **análise devastadora** do Expert v1 descobriu: o gargalo **não é** a quantidade de jogadas
(2M de self-play) — é a **diversidade de oponentes**. Self-play puro deixa margens finas contra
estilos que ele nunca viu (NitExploiter: só +22 bb/100) e sangra ~17 bb/100 nos blinds.

## O que a célula faz (ponta a ponta)
1. **Baixa um snapshot pinado** do PHH Dataset v3 (CC BY 4.0, [Zenodo 17136841](https://zenodo.org/records/17136841)) e amostra deterministicamente todos os estratos site×stake.
2. **Governa antes de ajustar**: HMAC-SHA256, tempo explícito com mapa de fuso auditado, componentes jogador×sessão e split temporal train/validation/test; falha fechado e ajusta os priors somente no train.
3. **Calibra dois perfis humanos** a partir da distribuição real: **fish** (solto-passivo) e **reg** (TAG) — limiares de força calibrados empiricamente.
4. **Treina o Expert v2** (MaskablePPO, mesas paralelas) contra um **POOL diverso**: snapshot de si + random + heurístico + **sondas de exploração** (maniac/station/nit) + **fish/reg humanos** — rotacionados a cada ciclo.
5. **v3 (lição do gate v2, medida):** maniac e nit **garantidos** em toda rotação do pool e o melhor checkpoint escolhido pela **régua composta** (heurístico+maniac+nit — a mesma do gate), com 2× passos.
6. **GATE DEVASTADOR**: bateria pareada v1 × v2, 30 blocos, Bonferroni, MDE, não inferioridade e controle negativo bilateral. Mesmo aprovado, publica apenas candidato imutável; promoção canônica exige revisão independente.

## Depois de promovido
Baixe `poker_expert.onnx` do repositório configurado em `HF_REPO_ID` e substitua
`backend/models/poker_expert.onnx`; rode a opção **[4] Validar** do launcher POKER.

*Fontes: Zenodo 10796885 (CC BY 4.0) · uoftcprg/phh-dataset · lições internas: notebook 05
(warm-start reprovado) e a auditoria devastadora (jul/2026).*


In [ ]:
# ============================================================
# EXPERT v2 "POPULACAO" - dados REAIS + pool de oponentes + gate devastador
# Celula UNICA: Run all. Secrets/config: GH_TOKEN, HF_TOKEN, HF_REPO_ID, PROJECT_COMMIT,
# HF_BASELINE_REVISION, PHH_PLAYER_HASH_KEY, PHH_HASH_KEY_ID, PHH_TIME_ZONE_MAP_JSON
# e POKER_CANDIDATE_LICENSE_ID/REFERENCE + POKER_CANDIDATE_LINEAGE_ID/REFERENCE.
#
# O QUE ESTA CELULA FAZ (ponta a ponta):
#  1. Baixa snapshot pinado do PHH v3 (CC BY 4.0) com hashes verificaveis
#  2. Extrai PRIORS equal-strata (nao representativos da populacao)
#  3. Calibra perfis humanos ("fish" solto-passivo, "reg" TAG) a partir dos dados
#  4. Treina o Expert v2 (PPO) contra um POOL DIVERSO: snapshot + regras + sondas
#     de exploracao (maniac/station/nit) + perfis humanos calibrados
#  5. GATE DEVASTADOR: avalia o v2; nunca o disponibiliza no runtime/Factory
# ============================================================
import base64
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from google.colab import userdata

os.environ['PYTHONHASHSEED'] = '20260717'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

gh = userdata.get('GH_TOKEN')
PROJECT_COMMIT = userdata.get('PROJECT_COMMIT')
HF_REPO_ID = userdata.get('HF_REPO_ID')
HF_BASELINE_REVISION = userdata.get('HF_BASELINE_REVISION')
PHH_PLAYER_HASH_KEY_TEXT = userdata.get('PHH_PLAYER_HASH_KEY')
PHH_HASH_KEY_ID = userdata.get('PHH_HASH_KEY_ID')
PHH_TIME_ZONE_MAP_JSON = userdata.get('PHH_TIME_ZONE_MAP_JSON')
PHH_GOVERNANCE_REVIEW_ID = userdata.get('POKER_PHH_GOVERNANCE_REVIEW_ID')
PHH_LICENSE_ID = userdata.get('POKER_PHH_LICENSE_ID')
PHH_LICENSE_STATUS = userdata.get('POKER_PHH_LICENSE_STATUS')
PHH_LICENSE_REFERENCE = userdata.get('POKER_PHH_LICENSE_REFERENCE')
PHH_ORIGIN_REFERENCE = userdata.get('POKER_PHH_ORIGIN_REFERENCE')
PHH_ORIGIN_STATUS = userdata.get('POKER_PHH_ORIGIN_STATUS')
PHH_TERMS_REFERENCE = userdata.get('POKER_PHH_TERMS_REFERENCE')
PHH_TEMPORAL_POLICY_REFERENCE = userdata.get('POKER_PHH_TEMPORAL_POLICY_REFERENCE')
PHH_TERMS_STATUS = userdata.get('POKER_PHH_TERMS_STATUS')
PHH_REAL_MONEY = userdata.get('POKER_PHH_REAL_MONEY')
PHH_PERMITTED_USE = userdata.get('POKER_PHH_PERMITTED_USE')
CANDIDATE_LICENSE_ID = userdata.get('POKER_CANDIDATE_LICENSE_ID')
CANDIDATE_LICENSE_REFERENCE = userdata.get('POKER_CANDIDATE_LICENSE_REFERENCE')
CANDIDATE_LINEAGE_ID = userdata.get('POKER_CANDIDATE_LINEAGE_ID')
CANDIDATE_LINEAGE_REFERENCE = userdata.get('POKER_CANDIDATE_LINEAGE_REFERENCE')
sensitive_env_name = re.compile(r'(?i)(^|_)(TOKEN|SECRET|PASSWORD|PASSWD|PRIVATE_KEY|CREDENTIALS?|API_KEY|ACCESS_KEY)($|_)')
for inherited_name in tuple(os.environ):
    if sensitive_env_name.search(inherited_name):
        os.environ.pop(inherited_name, None)
if not gh:
    raise RuntimeError('Configure GH_TOKEN para o clone pinado.')
HF_REPO_ID_PATTERN = r'[A-Za-z0-9](?:[A-Za-z0-9._-]{0,94}[A-Za-z0-9])?/[A-Za-z0-9](?:[A-Za-z0-9._-]{0,94}[A-Za-z0-9])?'
if (not isinstance(HF_REPO_ID, str) or not HF_REPO_ID.isascii()
        or len(HF_REPO_ID) > 96 or re.fullmatch(HF_REPO_ID_PATTERN, HF_REPO_ID) is None
        or '..' in HF_REPO_ID or '--' in HF_REPO_ID):
    raise RuntimeError('HF_REPO_ID deve ser ASCII owner/repo canonico.')
if (not isinstance(PROJECT_COMMIT, str)
        or re.fullmatch(r'[0-9a-f]{40}', PROJECT_COMMIT) is None):
    raise RuntimeError('PROJECT_COMMIT deve ser SHA Git canonico lowercase de 40 hex.')
if (not isinstance(HF_BASELINE_REVISION, str)
        or re.fullmatch(r'[0-9a-f]{40}', HF_BASELINE_REVISION) is None):
    raise RuntimeError('HF_BASELINE_REVISION deve ser commit HF lowercase de 40 hex.')
if not PHH_PLAYER_HASH_KEY_TEXT or len(PHH_PLAYER_HASH_KEY_TEXT.encode()) < 32:
    raise RuntimeError('PHH_PLAYER_HASH_KEY deve ter pelo menos 32 bytes.')
if not PHH_HASH_KEY_ID:
    raise RuntimeError('PHH_HASH_KEY_ID opaco e obrigatorio.')
if not PHH_TIME_ZONE_MAP_JSON:
    raise RuntimeError(
        'PHH_TIME_ZONE_MAP_JSON e obrigatorio; fusos ausentes nao sao inferidos.'
    )
for config_name, config_value in {
    'POKER_PHH_GOVERNANCE_REVIEW_ID': PHH_GOVERNANCE_REVIEW_ID,
    'POKER_PHH_LICENSE_ID': PHH_LICENSE_ID,
    'POKER_PHH_LICENSE_STATUS': PHH_LICENSE_STATUS,
    'POKER_PHH_LICENSE_REFERENCE': PHH_LICENSE_REFERENCE,
    'POKER_PHH_ORIGIN_REFERENCE': PHH_ORIGIN_REFERENCE,
    'POKER_PHH_ORIGIN_STATUS': PHH_ORIGIN_STATUS,
    'POKER_PHH_TERMS_REFERENCE': PHH_TERMS_REFERENCE,
    'POKER_PHH_TEMPORAL_POLICY_REFERENCE': PHH_TEMPORAL_POLICY_REFERENCE,
    'POKER_PHH_TERMS_STATUS': PHH_TERMS_STATUS,
    'POKER_PHH_REAL_MONEY': PHH_REAL_MONEY,
    'POKER_PHH_PERMITTED_USE': PHH_PERMITTED_USE,
    'POKER_CANDIDATE_LICENSE_ID': CANDIDATE_LICENSE_ID,
    'POKER_CANDIDATE_LICENSE_REFERENCE': CANDIDATE_LICENSE_REFERENCE,
    'POKER_CANDIDATE_LINEAGE_ID': CANDIDATE_LINEAGE_ID,
    'POKER_CANDIDATE_LINEAGE_REFERENCE': CANDIDATE_LINEAGE_REFERENCE,
}.items():
    if not isinstance(config_value, str) or not config_value.strip():
        raise RuntimeError(f'{config_name} deve ser fornecido pelo operador; nao sera inferido.')
if PHH_TERMS_STATUS != 'reviewed-compatible':
    raise RuntimeError('POKER_PHH_TERMS_STATUS deve ser reviewed-compatible.')
if PHH_LICENSE_STATUS != 'verified':
    raise RuntimeError('POKER_PHH_LICENSE_STATUS deve ser verified.')
if PHH_ORIGIN_STATUS != 'verified':
    raise RuntimeError('POKER_PHH_ORIGIN_STATUS deve ser verified.')
if PHH_REAL_MONEY != 'false':
    raise RuntimeError('Corpus com real_money diferente de false e proibido.')
if PHH_PERMITTED_USE != 'model-training':
    raise RuntimeError('POKER_PHH_PERMITTED_USE deve ser model-training.')
repo_workspace = Path(tempfile.mkdtemp(prefix='poker-source-')).resolve()
repo_dir = repo_workspace / 'poker-arena'
basic = base64.b64encode(f'x-access-token:{gh}'.encode()).decode()
git_executable = shutil.which('git')
if not git_executable:
    raise RuntimeError('git nao esta instalado no runtime.')
safe_env_names = ('PATH', 'HOME', 'LANG', 'LC_ALL', 'TMPDIR', 'SYSTEMROOT', 'WINDIR', 'USERPROFILE', 'TEMP', 'TMP')
git_env = {name: os.environ[name] for name in safe_env_names if name in os.environ}
clone_env = {**git_env,
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.extraHeader',
    'GIT_CONFIG_VALUE_0': f'Authorization: Basic {basic}',
}
subprocess.run(  # noqa: S603 - git resolvido e argv/URL fixos
    [git_executable, 'clone', '--quiet', 'https://github.com/FELIPEACASTRO/poker-arena.git', str(repo_dir)],
    check=True, env=clone_env,
)
subprocess.run(  # noqa: S603 - git resolvido e checkout pinado
    [git_executable, '-C', str(repo_dir), 'checkout', '--quiet', '--detach', PROJECT_COMMIT],
    check=True, env=git_env,
)
resolved_commit = subprocess.check_output(  # noqa: S603 - git resolvido e comando fixo
    [git_executable, '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True, env=git_env,
).strip()
if resolved_commit != PROJECT_COMMIT.lower():
    raise RuntimeError('checkout do projeto divergiu de PROJECT_COMMIT.')
working_tree = subprocess.check_output(  # noqa: S603 - git resolvido e checkout novo
    [git_executable, '-C', str(repo_dir), 'status', '--porcelain', '--untracked-files=all'],
    text=True, env=git_env,
).strip()
if working_tree:
    raise RuntimeError('checkout pinado contem alteracoes ou arquivos nao rastreados.')
pip_env = {**git_env, 'PIP_INDEX_URL': 'https://pypi.org/simple', 'PIP_CONFIG_FILE': os.devnull}
subprocess.run(  # noqa: S603 - sys.executable, checkout limpo e versoes exatas
    [sys.executable, '-m', 'pip', 'install', '-q', str(repo_dir / 'backend'),
     'stable-baselines3==2.7.0', 'sb3-contrib==2.7.0', 'gymnasium==1.2.1',
     'onnx==1.18.0', 'onnxruntime==1.27.0', 'huggingface_hub==0.34.3',
     'requests==2.34.2'],
    check=True, env=pip_env,
)
del basic, clone_env, git_env, pip_env
shutil.rmtree(repo_workspace)
hf_token = userdata.get('HF_TOKEN')
if not isinstance(hf_token, str) or not hf_token:
    raise RuntimeError('Configure HF_TOKEN depois da instalacao pinada.')
print('Instalado sem incluir credenciais na URL remota ou na saida.')

import hashlib
import hmac
import json
import math
import random
import time
from bisect import bisect_left, bisect_right
from collections import Counter, defaultdict
from datetime import date, datetime, time as datetime_time, timedelta, timezone
from itertools import combinations
from urllib.parse import quote, urlsplit
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

import numpy as np
import requests
import torch
import gymnasium as gym
import onnxruntime as ort
from gymnasium import spaces
from sb3_contrib import MaskablePPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import SubprocVecEnv
from huggingface_hub import HfApi, hf_hub_download

from poker_arena.bots.heuristic_bot import preflop_strength
from poker_arena.bots.ml_bot import EvaluationMLBot, MLBot
from poker_arena.bots.observation import Observation, observation_for
from poker_arena.engine.actions import Action, ActionType
from poker_arena.engine.cards import Deck
from poker_arena.engine.game import Hand
from poker_arena.engine.player import Player, PlayerStatus
from poker_arena.ml import FEATURE_SIZE, N_ACTIONS, SelfPlayEnv, encode, eval_bb100
from poker_arena.model_artifacts import (
    verify_evaluation_candidate, verify_model_artifact,
)

try:  # tomllib e stdlib no 3.11+
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib


def require_operator_reference(reference, label):
    if reference != reference.strip() or not reference.isascii():
        raise RuntimeError(f'{label} deve ser canonica e ASCII.')
    try:
        parsed = urlsplit(reference)
        port = parsed.port
    except ValueError as exc:
        raise RuntimeError(f'{label} invalida.') from exc
    host_labels = parsed.hostname.lower().split('.') if parsed.hostname else []
    valid_host = (len(host_labels) >= 2 and all(
        re.fullmatch(r'[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?', part)
        for part in host_labels
    ))
    valid_https = (parsed.scheme == 'https' and valid_host
                   and parsed.username is None and parsed.password is None
                   and port in (None, 443) and not parsed.query and not parsed.fragment
                   and parsed.path not in ('', '/') and parsed.path.startswith('/')
                   and '..' not in parsed.path.split('/'))
    valid_urn = (parsed.scheme == 'urn' and not parsed.netloc and not parsed.query
                 and not parsed.fragment and re.fullmatch(
                     r"urn:[a-z0-9][a-z0-9-]{1,31}:[A-Za-z0-9][A-Za-z0-9._~:/@!$&'()*+,;=%-]{0,1023}",
                     reference,
                 ) is not None)
    if not (valid_https or valid_urn):
        raise RuntimeError(f'{label} deve ser HTTPS absoluto ou URN.')


def require_operator_governance(identifier, reference, label):
    if re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._:+/-]{0,127}', identifier) is None:
        raise RuntimeError(f'{label}_ID invalido; forneca um ID auditavel.')
    require_operator_reference(reference, f'{label}_REFERENCE')


require_operator_governance(
    CANDIDATE_LICENSE_ID, CANDIDATE_LICENSE_REFERENCE, 'POKER_CANDIDATE_LICENSE',
)
require_operator_governance(
    CANDIDATE_LINEAGE_ID, CANDIDATE_LINEAGE_REFERENCE, 'POKER_CANDIDATE_LINEAGE',
)
require_operator_governance(
    PHH_LICENSE_ID, PHH_LICENSE_REFERENCE, 'POKER_PHH_LICENSE',
)
if re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._:+/-]{0,127}', PHH_GOVERNANCE_REVIEW_ID) is None:
    raise RuntimeError('POKER_PHH_GOVERNANCE_REVIEW_ID invalido.')
for reference, label in (
    (PHH_ORIGIN_REFERENCE, 'POKER_PHH_ORIGIN_REFERENCE'),
    (PHH_TERMS_REFERENCE, 'POKER_PHH_TERMS_REFERENCE'),
    (PHH_TEMPORAL_POLICY_REFERENCE, 'POKER_PHH_TEMPORAL_POLICY_REFERENCE'),
):
    require_operator_reference(reference, label)


# ---------------- logs estilo Use a Cabeca ----------------
def box(titulo, linhas=()):
    w = max([len(titulo)] + [len(s) for s in linhas]) if linhas else len(titulo)
    print('\n  +' + '-' * (w + 2) + '+')
    print('  | ' + titulo.ljust(w) + ' |')
    print('  +' + '-' * (w + 2) + '+')
    for s in linhas:
        print('  | ' + s.ljust(w) + ' |')
    print('  +' + '-' * (w + 2) + '+')


def barra(frac, n=22):
    frac = max(0.0, min(1.0, frac))
    c = int(frac * n)
    return '[' + '#' * c + '.' * (n - c) + f'] {100 * frac:3.0f}%'


# ---------------- CONFIG ----------------
PLAYER_IDENTITY_SCHEME = 'hmac-sha256'
PLAYER_IDENTITY_KEY_VERSION = PHH_HASH_KEY_ID
PLAYER_IDENTITY_SEPARATION_CONTEXT = 'poker-arena/phh-v3/notebook-07/v1'
PLAYER_IDENTITY_PURPOSE = 'split-leakage-prevention'
if re.fullmatch(r'kv-[A-Za-z0-9][A-Za-z0-9._-]{2,62}', PLAYER_IDENTITY_KEY_VERSION) is None:
    raise RuntimeError('PHH_HASH_KEY_ID deve ser um ID opaco no formato kv-*, nunca a chave.')
PHH_PLAYER_HASH_KEY = bytearray(PHH_PLAYER_HASH_KEY_TEXT.encode('utf-8'))
PHH_PLAYER_HASH_KEY_TEXT = ''
del PHH_PLAYER_HASH_KEY_TEXT
PHH_TIME_ZONE_MAP = json.loads(PHH_TIME_ZONE_MAP_JSON)
if not isinstance(PHH_TIME_ZONE_MAP, dict) or not PHH_TIME_ZONE_MAP:
    raise RuntimeError('PHH_TIME_ZONE_MAP_JSON deve ser um objeto JSON nao vazio.')
for mapping_key, zone_spec in PHH_TIME_ZONE_MAP.items():
    if (not isinstance(mapping_key, str) or not mapping_key
            or not isinstance(zone_spec, str) or not zone_spec):
        raise RuntimeError('Mapa de fusos deve conter apenas strings nao vazias.')
PHH_TIME_ZONE_MAP_CANONICAL = json.dumps(
    PHH_TIME_ZONE_MAP, sort_keys=True, separators=(',', ':'), ensure_ascii=True,
)
PHH_TIME_ZONE_MAP_SHA256 = hashlib.sha256(
    PHH_TIME_ZONE_MAP_CANONICAL.encode('utf-8')
).hexdigest()
del PHH_TIME_ZONE_MAP_JSON, PHH_TIME_ZONE_MAP_CANONICAL
INGESTION_NOW_UTC = datetime.now(timezone.utc)
MAX_FUTURE_CLOCK_SKEW = timedelta(minutes=5)
SPLIT_TARGETS = {'train': 0.70, 'validation': 0.15, 'test': 0.15}
MIN_HANDS_PER_SPLIT = 1_000
STRATUM_DIR_RE = re.compile(
    r'(?P<site>[A-Z]+)-(?P<start>\d{4}-\d{2}-\d{2})_'
    r'(?P<end>\d{4}-\d{2}-\d{2})_'
    r'(?P<stake>25|50|100|200|400|600|1000)NLH_OBFU'
)
DATA_REPO = 'uoftcprg/phh-dataset'
DATA_COMMIT = 'e2ec038d31a1a46a82d147db4bbfdb0910459705'  # tag oficial v3
DATA_TREE = 'a1c33bedd561407860b2467bffb4257317e9d6ce'
DATA_SAMPLE_SEED = 20260717
SAMPLE_HANDS = 250_000        # maos reais a parsear (de 21.6M) - de sobra p/ priors
MIN_HANDS_PER_PLAYER = 50     # so opina sobre jogador com amostra decente
TRAINING_SEED = 20260717
N_ENVS = 8
TOTAL_STEPS = 3_000_000       # v3: 2x passos (a run v2 ainda oscilava no fim)
EVAL_EVERY = 200_000
GATE_HANDS = 400              # maos por bloco pareado
GATE_SEEDS = tuple(range(10_000, 10_030))  # 30 blocos independentes, fora do treino
CONTROL_SEEDS = tuple(range(20_000, 20_030))  # painel baseline independente
CHECKPOINT_SELECTION_SEEDS = tuple(range(30_000, 30_005))
CHECKPOINT_SELECTION_HANDS = 300
GATE_REQUIRED_BLOCKS = 30
if not (set(CHECKPOINT_SELECTION_SEEDS).isdisjoint(GATE_SEEDS)
        and set(CHECKPOINT_SELECTION_SEEDS).isdisjoint(CONTROL_SEEDS)):
    raise RuntimeError('painel de selecao deve ser disjunto dos paineis finais.')
random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)
torch.use_deterministic_algorithms(True)
GATE_POWER_ANALYSIS_ID = os.environ.get('POKER_GATE_POWER_ANALYSIS_ID', '').strip()


def optional_float_env(name):
    raw = os.environ.get(name, '').strip()
    try:
        return float(raw) if raw else math.nan
    except ValueError:
        return math.nan


GATE_MDE_BB100 = optional_float_env('POKER_GATE_MDE_BB100')
GATE_NONINFERIORITY_MARGIN_BB100 = optional_float_env(
    'POKER_GATE_NONINFERIORITY_MARGIN_BB100'
)
GATE_CONTROL_EQUIVALENCE_MARGIN_BB100 = optional_float_env(
    'POKER_GATE_CONTROL_EQUIVALENCE_MARGIN_BB100'
)
BASELINE_MODEL_FILENAME = 'poker_expert.onnx'
BASELINE_MANIFEST_FILENAME = 'MANIFEST.json'
PUBLISH_ARTIFACTS = os.environ.get('POKER_PUBLISH_ARTIFACTS') == '1'

box('PREFLIGHT DO BASELINE GOVERNADO', [
    f'{HF_REPO_ID}@{HF_BASELINE_REVISION}',
    'download de modelo + manifesto antes de dados e treino',
])
V1_PATH = hf_hub_download(
    HF_REPO_ID, BASELINE_MODEL_FILENAME, revision=HF_BASELINE_REVISION,
    token=hf_token,
)
V1_MANIFEST_PATH = hf_hub_download(
    HF_REPO_ID, BASELINE_MANIFEST_FILENAME, revision=HF_BASELINE_REVISION,
    token=hf_token,
)
hf_token = ''
baseline_artifact = verify_model_artifact(
    V1_PATH, 'expert', manifest_path=V1_MANIFEST_PATH,
)
if baseline_artifact.usage != 'deployment':
    raise RuntimeError('baseline nao recebeu autorizacao de deployment.')
print(f'  baseline governado: {baseline_artifact.sha256}')

box('EXPERT v2 POPULACAO - o plano', [
    'Descoberta da analise devastadora: o gargalo NAO e quantidade de',
    'jogadas (2M de self-play) e sim DIVERSIDADE de oponentes.',
    '',
    '1. maos REAIS (Zenodo, CC BY 4.0)  -> priors do Adaptativo',
    '2. perfis humanos calibrados       -> fish (solto) e reg (TAG)',
    '3. treino vs POOL diverso          -> snapshot+regras+sondas+humanos',
    '4. gate pareado -> publica somente candidate; promocao e externa',
])

# ============================================================
# [1/5] DADOS REAIS - amostra do handhq direto do GitHub (sem zip de 20GB!)
# ============================================================
box('[1/5] Localizando maos reais (handhq - dados do Zenodo 10796885, CC BY 4.0)')
HDRS = {'Authorization': f'Bearer {gh}', 'Accept': 'application/vnd.github+json'}
github_session = requests.Session()
github_session.trust_env = False
commit_response = None
tree_response = None
try:
    commit_response = github_session.get(
        f'https://api.github.com/repos/{DATA_REPO}/git/commits/{DATA_COMMIT}',
        headers=HDRS, timeout=60,
    )
    commit_response.raise_for_status()
    commit_payload = commit_response.json()
    if commit_payload.get('sha') != DATA_COMMIT:
        raise RuntimeError('GitHub retornou commit diferente do snapshot pinado.')
    if commit_payload.get('tree', {}).get('sha') != DATA_TREE:
        raise RuntimeError('GitHub retornou tree diferente do snapshot pinado.')
    tree_response = github_session.get(
        f'https://api.github.com/repos/{DATA_REPO}/git/trees/{DATA_TREE}',
        params={'recursive': '1'}, headers=HDRS, timeout=120,
    )
    tree_response.raise_for_status()
    tree_payload = tree_response.json()
    if tree_payload.get('sha') != DATA_TREE:
        raise RuntimeError('Git tree diferente do snapshot esperado.')
    if tree_payload.get('truncated') is not False:
        raise RuntimeError('Git tree truncada; amostragem bloqueada.')
except requests.RequestException:
    raise RuntimeError('consulta governada ao GitHub falhou') from None
finally:
    for response in (commit_response, tree_response):
        if response is not None:
            response.request.headers.pop('Authorization', None)
            response.close()
    HDRS.clear()
    gh = ''
    github_session.close()
del gh, HDRS, github_session, commit_response, tree_response
entries = [
    item for item in tree_payload.get('tree', [])
    if item.get('type') == 'blob'
    and item.get('path', '').startswith('data/handhq/')
    and item.get('path', '').lower().endswith(('.phh', '.phhs'))
]
if not entries:
    raise RuntimeError('nenhum PHH encontrado no snapshot pinado.')

by_stratum = defaultdict(list)
for item in entries:
    parts = item['path'].split('/')
    if len(parts) < 4:
        raise RuntimeError(f'path PHH inesperado: {item["path"]}')
    directory_match = STRATUM_DIR_RE.fullmatch(parts[2])
    if directory_match is None:
        raise RuntimeError(f'diretorio PHH fora do contrato v3: {parts[2]}')
    site = directory_match.group('site')
    stratum = f"{site}-{directory_match.group('stake')}NLH"
    governed_item = dict(item)
    governed_item.update({
        'site': site,
        'source_start': date.fromisoformat(directory_match.group('start')),
        'source_end': date.fromisoformat(directory_match.group('end')),
    })
    by_stratum[stratum].append(governed_item)

strata = sorted(by_stratum)
if len(strata) != 27:
    raise RuntimeError(f'esperava 27 estratos no PHH v3, encontrei {len(strata)}')
for stratum in strata:
    by_stratum[stratum].sort(
        key=lambda item: hashlib.sha256(
            f'{DATA_SAMPLE_SEED}|{item["path"]}'.encode()
        ).digest()
    )

# Round-robin deterministico entre todos os 27 site x stake; cobertura nao implica
# representatividade populacional, portanto os priors sao rotulados equal-strata.
phh_items = []
for file_index in range(max(map(len, by_stratum.values()))):
    for stratum in strata:
        if file_index < len(by_stratum[stratum]):
            phh_items.append((stratum, by_stratum[stratum][file_index]))
quota_base, quota_extra = divmod(SAMPLE_HANDS, len(strata))
stratum_quota = {
    stratum: quota_base + (index < quota_extra)
    for index, stratum in enumerate(strata)
}
print(f'  snapshot: {DATA_COMMIT}; {len(entries):,} arquivos; {len(strata)} estratos')
print(f'  alvo equal-strata: {SAMPLE_HANDS:,} maos ({quota_base:,} ou {quota_base + 1:,}/estrato)')

# ============================================================
# [2/5] PRIORS populacionais (parse PHH -> stats por jogador)
# ============================================================
box('[2/5] Baixando e parseando o comportamento humano REAL')

STAT_FIELDS = ('hands', 'vpip', 'pfr', 'faced', 'folded', 'calls', 'raises')
hand_records = []
dsu_parent = []
dsu_size = []
player_owner = {}
session_owner = {}
seen_hand_ids = set()
timezone_mapping_keys_used = set()
parsed = 0
stratum_parsed = Counter()
source_files = []
t0 = time.time()


def opaque_id(kind, raw_value):
    payload = (
        f'{PLAYER_IDENTITY_SEPARATION_CONTEXT}\0{kind}\0{raw_value}'
    ).encode('utf-8')
    return hmac.new(PHH_PLAYER_HASH_KEY, payload, hashlib.sha256).hexdigest()


def dsu_add():
    index = len(dsu_parent)
    dsu_parent.append(index)
    dsu_size.append(1)
    return index


def dsu_find(index):
    while dsu_parent[index] != index:
        dsu_parent[index] = dsu_parent[dsu_parent[index]]
        index = dsu_parent[index]
    return index


def dsu_union(left, right):
    left, right = dsu_find(left), dsu_find(right)
    if left == right:
        return left
    if dsu_size[left] < dsu_size[right]:
        left, right = right, left
    dsu_parent[right] = left
    dsu_size[left] += dsu_size[right]
    return left


def resolve_zone(zone_spec):
    fixed = re.fullmatch(r'([+-])(\d{2}):(\d{2})', zone_spec)
    if fixed is not None:
        hours, minutes = int(fixed.group(2)), int(fixed.group(3))
        if minutes >= 60 or hours > 14 or (hours == 14 and minutes):
            raise RuntimeError(f'offset UTC invalido no mapa: {zone_spec}')
        delta = timedelta(hours=hours, minutes=minutes)
        return timezone(delta if fixed.group(1) == '+' else -delta)
    try:
        return ZoneInfo(zone_spec)
    except ZoneInfoNotFoundError as exc:
        raise RuntimeError(f'fuso IANA desconhecido no mapa: {zone_spec}') from exc


def localize_without_guessing(local_naive, zone):
    if not isinstance(zone, ZoneInfo):
        return local_naive.replace(tzinfo=zone)
    candidates = {}
    for fold in (0, 1):
        aware = local_naive.replace(tzinfo=zone, fold=fold)
        round_trip = aware.astimezone(timezone.utc).astimezone(zone)
        if round_trip.replace(tzinfo=None) == local_naive:
            candidates[aware.utcoffset()] = aware
    if not candidates:
        raise RuntimeError('horario local inexistente por transicao de DST')
    if len(candidates) != 1:
        raise RuntimeError('horario local ambiguo por transicao de DST; fold ausente')
    return next(iter(candidates.values()))


def governed_context(raw, stratum, item):
    site = item['site']
    if not stratum.startswith(f'{site}-'):
        raise RuntimeError('site e estrato divergentes')
    date_fields = {name: raw.get(name) for name in ('year', 'month', 'day')}
    if any(isinstance(value, bool) or not isinstance(value, int) for value in date_fields.values()):
        raise RuntimeError('PHH sem year/month/day inteiros; data nao sera inventada')
    local_day = date(date_fields['year'], date_fields['month'], date_fields['day'])
    if not item['source_start'] <= local_day <= item['source_end']:
        raise RuntimeError('data explicita da mao fora do periodo declarado no snapshot')
    local_clock = raw.get('time')
    if not isinstance(local_clock, datetime_time) or local_clock.tzinfo is not None:
        raise RuntimeError('PHH sem horario local valido; event_time nao sera inventado')
    declared_zone = raw.get('time_zone')
    if declared_zone is not None:
        if not isinstance(declared_zone, str) or not declared_zone.strip():
            raise RuntimeError('time_zone PHH invalido')
        zone_spec = declared_zone.strip()
    else:
        abbreviation = raw.get('time_zone_abbreviation')
        if abbreviation is not None and (
                not isinstance(abbreviation, str) or not abbreviation.strip()):
            raise RuntimeError('time_zone_abbreviation PHH invalida')
        mapping_key = f"{site}|{abbreviation.strip() if abbreviation else '<missing>'}"
        if mapping_key not in PHH_TIME_ZONE_MAP:
            raise RuntimeError(
                f'fuso nao resolvido para {mapping_key}; forneca mapeamento auditado'
            )
        timezone_mapping_keys_used.add(mapping_key)
        zone_spec = PHH_TIME_ZONE_MAP[mapping_key]
    local_naive = datetime.combine(local_day, local_clock)
    event_time = localize_without_guessing(local_naive, resolve_zone(zone_spec))
    event_time = event_time.astimezone(timezone.utc)
    if event_time > INGESTION_NOW_UTC + MAX_FUTURE_CLOCK_SKEW:
        raise RuntimeError('event_time materialmente futuro; ingestao bloqueada')
    venue = raw.get('venue')
    table = raw.get('table')
    hand = raw.get('hand')
    if not isinstance(venue, str) or not venue.strip():
        raise RuntimeError('venue ausente; session nao pode ser derivada')
    if isinstance(table, bool) or not isinstance(table, (str, int)) or not str(table).strip():
        raise RuntimeError('table ausente; session nao pode ser derivada')
    if isinstance(hand, bool) or not isinstance(hand, (str, int)) or not str(hand).strip():
        raise RuntimeError('hand ausente; identidade da mao nao pode ser derivada')
    session = opaque_id(
        'session', f'{site}\x1f{venue.strip()}\x1f{str(table).strip()}\x1f{local_day.isoformat()}',
    )
    hand_id = opaque_id(
        'hand', f'{site}\x1f{venue.strip()}\x1f{str(hand).strip()}',
    )
    return event_time, session, hand_id


def eat_hand(raw, stratum, item):
    """PHH -> registro pseudonimizado; nenhum prior e ajustado nesta etapa."""
    players = raw.get('players')
    if not isinstance(players, list) or not all(isinstance(p, str) and p for p in players):
        raise RuntimeError('players ausente/invalido; hashed_player_id nao pode ser derivado')
    if len(set(players)) != len(players):
        raise RuntimeError('identidades duplicadas na mesma mao')
    n = len(players)
    if not 2 <= n <= 9 or len(raw.get('starting_stacks', [])) != n:
        raise RuntimeError('numero de jogadores/stacks fora do contrato PHH')
    site = item['site']
    hashed_player_ids = tuple(
        opaque_id('player', f'{site}\x1f{name}') for name in players
    )
    if len(set(hashed_player_ids)) != n or not all(
            re.fullmatch(r'[0-9a-f]{64}', player_id) for player_id in hashed_player_ids):
        raise RuntimeError('falha na pseudonimizacao HMAC de jogador')
    player_deltas = [[1, 0, 0, 0, 0, 0, 0] for _ in range(n)]
    blinds = list(raw.get('blinds_or_straddles') or [0] * n)
    cur = blinds[:n] + [0] * max(0, n - len(blinds))
    street_max = max(cur) if cur else 0
    street = 0
    vpip_seats, pfr_seats = set(), set()
    actions = raw.get('actions')
    if not isinstance(actions, list) or not all(isinstance(tok, str) for tok in actions):
        raise RuntimeError('actions PHH deve ser uma lista de strings.')
    for tok in actions:
        parts = tok.split()
        if not parts:
            raise RuntimeError('token PHH vazio.')
        if parts[0] == 'd':
            if len(parts) < 2 or parts[1] not in {'dh', 'db'}:
                raise RuntimeError(f'token de deal PHH invalido: {tok!r}')
            if parts[1] == 'db':
                if len(parts) != 3:
                    raise RuntimeError(f'token de board PHH invalido: {tok!r}')
                street += 1
                cur = [0] * n
                street_max = 0
            continue
        if len(parts) < 2 or re.fullmatch(r'p[1-9][0-9]*', parts[0]) is None:
            raise RuntimeError(f'token de jogador PHH invalido: {tok!r}')
        seat = int(parts[0][1:]) - 1
        if not 0 <= seat < n:
            raise RuntimeError(f'assento PHH fora dos limites: {seat}')
        verb = parts[1]
        if verb == 'sm':
            if len(parts) not in {2, 3}:
                raise RuntimeError(f'token sm PHH invalido: {tok!r}')
            continue
        if verb not in {'f', 'cc', 'cbr'}:
            raise RuntimeError(f'verbo PHH desconhecido: {verb!r}')
        st = player_deltas[seat]
        owe = max(0, street_max - cur[seat])
        if owe > 0:
            st[3] += 1
        if verb == 'f':
            if len(parts) != 2:
                raise RuntimeError(f'token fold PHH invalido: {tok!r}')
            if owe > 0:
                st[4] += 1
        elif verb == 'cc':
            if len(parts) != 2:
                raise RuntimeError(f'token call/check PHH invalido: {tok!r}')
            cur[seat] = street_max
            if owe > 0:  # check nao e call e nao entra no fator de agressao
                st[5] += 1
            if street == 0 and owe > 0:
                vpip_seats.add(seat)
        elif verb == 'cbr':
            if len(parts) != 3:
                raise RuntimeError(f'token raise PHH invalido: {tok!r}')
            to = float(parts[2])
            if not math.isfinite(to) or to <= street_max:
                raise RuntimeError(f'raise PHH nao crescente: {tok!r}')
            cur[seat] = to
            street_max = max(street_max, to)
            st[6] += 1
            if street == 0:
                vpip_seats.add(seat)
                pfr_seats.add(seat)
    for seat in vpip_seats:
        player_deltas[seat][1] = 1
    for seat in pfr_seats:
        player_deltas[seat][2] = 1
    event_time, session, hand_id = governed_context(raw, stratum, item)
    if hand_id in seen_hand_ids:
        raise RuntimeError('mao duplicada no snapshot amostrado')
    seen_hand_ids.add(hand_id)
    index = dsu_add()
    hand_records.append({
        'hand_id': hand_id, 'event_time': event_time, 'session': session,
        'split_group': None, 'hashed_player_id': hashed_player_ids,
        'player_deltas': tuple(tuple(delta) for delta in player_deltas),
        'stratum': stratum,
    })
    previous_session = session_owner.get(session)
    if previous_session is None:
        session_owner[session] = index
    else:
        dsu_union(index, previous_session)
    for player_id in hashed_player_ids:
        previous_player = player_owner.get(player_id)
        if previous_player is None:
            player_owner[player_id] = index
        else:
            dsu_union(index, previous_player)


def hands_in(blob):
    """.phh = 1 mao por arquivo; .phhs = varias maos como secoes TOML [1],[2],..."""
    data = tomllib.loads(blob)
    if 'actions' in data:
        return [data]
    hands = [v for v in data.values() if isinstance(v, dict) and 'actions' in v]
    if not hands:
        raise RuntimeError('arquivo PHH sem maos reconheciveis.')
    return hands


for stratum, item in phh_items:
    if stratum_parsed[stratum] >= stratum_quota[stratum]:
        continue
    path = item['path']
    url = f'https://raw.githubusercontent.com/{DATA_REPO}/{DATA_COMMIT}/{quote(path, safe="/")}'
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    content = response.content
    if len(content) != item['size'] or len(content) > 20 * 1024 * 1024:
        raise RuntimeError('blob PHH diverge do tamanho pinado ou excede 20 MiB.')
    git_blob = hashlib.sha1(f'blob {len(content)}\0'.encode() + content, usedforsecurity=False).hexdigest()
    if git_blob != item['sha']:
        raise RuntimeError('conteudo diverge da Git tree pinada.')
    content_sha256 = hashlib.sha256(content).hexdigest()
    used_from_file = 0
    blob = content.decode('utf-8')
    for raw in hands_in(blob):
        if stratum_parsed[stratum] >= stratum_quota[stratum]:
            break
        # O alvo e ring game; HU/3p nao entra silenciosamente na distribuicao.
        if 'starting_stacks' in raw and len(raw['starting_stacks']) >= 4:
            eat_hand(raw, stratum, item)
            parsed += 1
            used_from_file += 1
            stratum_parsed[stratum] += 1
            if parsed % 50_000 == 0:
                sp = parsed / max(time.time() - t0, 1e-9)
                print(f'  {barra(parsed / SAMPLE_HANDS)}  {sp:.0f} maos/s')
    if used_from_file:
        source_files.append({
            'path': path, 'git_blob_sha1': item['sha'],
            'sha256': content_sha256, 'hands_used': used_from_file,
            'stratum': stratum,
        })

if parsed != SAMPLE_HANDS or len(hand_records) != SAMPLE_HANDS:
    raise RuntimeError(f'amostra incompleta: {parsed}/{SAMPLE_HANDS}')
if not all(stratum_parsed[s] == stratum_quota[s] for s in strata):
    raise RuntimeError(f'quota equal-strata incompleta: {stratum_parsed}')
for key_index in range(len(PHH_PLAYER_HASH_KEY)):
    PHH_PLAYER_HASH_KEY[key_index] = 0
PHH_PLAYER_HASH_KEY.clear()
del key_index

# Componentes conexos unem qualquer mao que compartilhe jogador ou sessao.
# Assim, um jogador/sessao jamais pode atravessar o corte, mesmo indiretamente.
component_indices = defaultdict(list)
for record_index in range(len(hand_records)):
    component_indices[dsu_find(record_index)].append(record_index)
components = []
for indices in component_indices.values():
    group_material = '\n'.join(sorted(hand_records[i]['hand_id'] for i in indices))
    split_group = hashlib.sha256(
        f'{PLAYER_IDENTITY_SEPARATION_CONTEXT}\0split_group\0{group_material}'.encode()
    ).hexdigest()
    for record_index in indices:
        hand_records[record_index]['split_group'] = split_group
    event_times = [hand_records[i]['event_time'] for i in indices]
    components.append({
        'indices': tuple(indices), 'split_group': split_group,
        'min_event_time': min(event_times), 'max_event_time': max(event_times),
        'hands': len(indices),
    })
if len(components) < 3:
    raise RuntimeError('grafo jogador/sessao nao permite tres splits independentes')
components.sort(key=lambda component: (
    component['min_event_time'], component['max_event_time'], component['split_group'],
))

prefix_hands = [0]
prefix_max_event_time = []
running_max = None
for component in components:
    prefix_hands.append(prefix_hands[-1] + component['hands'])
    running_max = (
        component['max_event_time'] if running_max is None
        else max(running_max, component['max_event_time'])
    )
    prefix_max_event_time.append(running_max)
valid_cuts = [
    position for position in range(1, len(components))
    if prefix_max_event_time[position - 1] < components[position]['min_event_time']
]
if len(valid_cuts) < 2:
    raise RuntimeError('nao existem dois cortes temporais estritos entre componentes')

# Escolha deterministica mais proxima de 70/15/15 entre cortes temporais validos.
valid_counts = [prefix_hands[position] for position in valid_cuts]
best_pair = None
for train_position, train_cut in enumerate(valid_cuts):
    train_hands = prefix_hands[train_cut]
    if train_hands < MIN_HANDS_PER_SPLIT:
        continue
    lower = bisect_left(
        valid_counts, train_hands + MIN_HANDS_PER_SPLIT, train_position + 1,
    )
    upper = bisect_right(
        valid_counts, parsed - MIN_HANDS_PER_SPLIT, train_position + 1,
    )
    if lower >= upper:
        continue
    target_validation_end = int(parsed * (SPLIT_TARGETS['train'] + SPLIT_TARGETS['validation']))
    pivot = bisect_left(valid_counts, target_validation_end, lower, upper)
    for validation_position in sorted({pivot - 1, pivot}):
        if not lower <= validation_position < upper:
            continue
        validation_cut = valid_cuts[validation_position]
        validation_hands = prefix_hands[validation_cut] - train_hands
        test_hands = parsed - prefix_hands[validation_cut]
        score = (
            abs(train_hands / parsed - SPLIT_TARGETS['train'])
            + abs(validation_hands / parsed - SPLIT_TARGETS['validation'])
            + abs(test_hands / parsed - SPLIT_TARGETS['test'])
        )
        candidate = (score, train_cut, validation_cut)
        if best_pair is None or candidate < best_pair:
            best_pair = candidate
if best_pair is None:
    raise RuntimeError('nenhum split temporal agrupado satisfaz os tamanhos minimos')
_, train_cut, validation_cut = best_pair
for position, component in enumerate(components):
    split = (
        'train' if position < train_cut
        else 'validation' if position < validation_cut
        else 'test'
    )
    for record_index in component['indices']:
        hand_records[record_index]['split'] = split

split_names = ('train', 'validation', 'test')
records_by_split = {
    split: [record for record in hand_records if record['split'] == split]
    for split in split_names
}
identity_sets = {}
for split, records in records_by_split.items():
    identity_sets[split] = {
        'hashed_player_id': {
            player_id for record in records for player_id in record['hashed_player_id']
        },
        'session': {record['session'] for record in records},
        'split_group': {record['split_group'] for record in records},
    }
overlap_counts = {field: {} for field in ('hashed_player_id', 'session', 'split_group')}
for left, right in combinations(split_names, 2):
    for field in overlap_counts:
        overlap_counts[field][f'{left}__{right}'] = len(
            identity_sets[left][field] & identity_sets[right][field]
        )
if any(count for field_counts in overlap_counts.values() for count in field_counts.values()):
    raise RuntimeError(f'vazamento de identidade/grupo entre splits: {overlap_counts}')
split_time_ranges = {
    split: (
        min(record['event_time'] for record in records),
        max(record['event_time'] for record in records),
    )
    for split, records in records_by_split.items()
}
strict_temporal_order = {
    'train_before_validation': split_time_ranges['train'][1] < split_time_ranges['validation'][0],
    'validation_before_test': split_time_ranges['validation'][1] < split_time_ranges['test'][0],
}
if not all(strict_temporal_order.values()):
    raise RuntimeError(f'ordem temporal estrita violada: {strict_temporal_order}')

def utc_iso(value):
    return value.astimezone(timezone.utc).isoformat().replace('+00:00', 'Z')


assignment_hasher = hashlib.sha256()
for record in sorted(hand_records, key=lambda row: row['hand_id']):
    assignment_hasher.update(
        f"{record['hand_id']}|{record['split']}|{record['split_group']}|"
        f"{utc_iso(record['event_time'])}\n".encode('ascii')
    )
split_summary = {}
for split, records in records_by_split.items():
    split_summary[split] = {
        'hands': len(records),
        'fraction': len(records) / parsed,
        'players': len(identity_sets[split]['hashed_player_id']),
        'sessions': len(identity_sets[split]['session']),
        'split_groups': len(identity_sets[split]['split_group']),
        'event_time_min': utc_iso(split_time_ranges[split][0]),
        'event_time_max': utc_iso(split_time_ranges[split][1]),
        'stratum_hands': dict(sorted(Counter(
            record['stratum'] for record in records
        ).items())),
    }
split_receipt = {
    'schema_version': 1,
    'contract': 'poker-arena.phh-split/v1',
    'dataset': {
        'repository': DATA_REPO, 'commit': DATA_COMMIT, 'tree': DATA_TREE,
        'origin_status': PHH_ORIGIN_STATUS,
        'sampling_seed': DATA_SAMPLE_SEED, 'sample_hands': parsed,
    },
    'required_fields': ['hashed_player_id', 'event_time', 'session', 'split_group'],
    'event_time_source': (
        'explicit PHH year/month/day/time plus PHH time_zone or operator-audited map; '
        'no filename, mtime, sequence, or synthetic-date fallback'
    ),
    'player_identity': {
        'scheme': PLAYER_IDENTITY_SCHEME,
        'key_version': PLAYER_IDENTITY_KEY_VERSION,
        'separation_context': PLAYER_IDENTITY_SEPARATION_CONTEXT,
        'purpose': PLAYER_IDENTITY_PURPOSE,
    },
    'timezone_resolution': {
        'operator_mapping_sha256': PHH_TIME_ZONE_MAP_SHA256,
        'mapping_keys_used': sorted(timezone_mapping_keys_used),
    },
    'split_algorithm': 'player-session-connected-components-temporal-v1',
    'target_fractions': SPLIT_TARGETS,
    'minimum_hands_per_split': MIN_HANDS_PER_SPLIT,
    'overlap_counts': overlap_counts,
    'strict_temporal_order': strict_temporal_order,
    'splits': split_summary,
    'assignment_sha256': assignment_hasher.hexdigest(),
    'prior_fit_split': 'train',
    'held_out_policy': 'validation/test are excluded from prior estimation and PPO training',
}
split_receipt_bytes = json.dumps(
    split_receipt, sort_keys=True, separators=(',', ':'), ensure_ascii=True,
).encode('utf-8')
split_receipt_sha256 = hashlib.sha256(split_receipt_bytes).hexdigest()
with open('phh_split_receipt.json', 'w', encoding='utf-8') as receipt_file:
    json.dump(split_receipt, receipt_file, indent=2, sort_keys=True)

# Recibo especifico do corpus PHH. O DATA_MANIFEST v1 generico modela uma
# entidade/jogador por arquivo; uma mao PHH e multi-player, portanto declarar
# conformidade com aquele schema seria uma falsificacao de granularidade.
phh_corpus_governance_receipt = {
    'schema_version': 1,
    'contract': 'poker-arena.phh-corpus-governance/v1',
    'operator_review': {'id': PHH_GOVERNANCE_REVIEW_ID},
    'dataset': {
        'repository': DATA_REPO, 'commit': DATA_COMMIT, 'tree': DATA_TREE,
        'origin_reference': PHH_ORIGIN_REFERENCE,
        'origin_status': PHH_ORIGIN_STATUS,
        'unit_of_analysis': 'multiplayer-hand',
    },
    'license': {
        'status': PHH_LICENSE_STATUS, 'id': PHH_LICENSE_ID,
        'reference': PHH_LICENSE_REFERENCE,
    },
    'platform_terms': {
        'status': PHH_TERMS_STATUS, 'reference': PHH_TERMS_REFERENCE,
    },
    'policy': {
        'real_money': False, 'permitted_use': ['model-training'],
        'player_identity_purpose': PLAYER_IDENTITY_PURPOSE,
    },
    'temporal': {
        'policy_reference': PHH_TEMPORAL_POLICY_REFERENCE,
        'split_receipt_sha256': split_receipt_sha256,
        'strict_order_verified': all(strict_temporal_order.values()),
        'event_time_source': split_receipt['event_time_source'],
    },
    'generic_dataset_manifest_v1': {
        'validated': False,
        'reason': (
            'not representable without falsifying entity-per-file semantics; PHH files '
            'contain multiplayer hands; this corpus-specific receipt is not equivalence'
        ),
    },
}
if (phh_corpus_governance_receipt['license']['status'] != 'verified'
        or phh_corpus_governance_receipt['dataset']['origin_status'] != 'verified'
        or phh_corpus_governance_receipt['platform_terms']['status'] != 'reviewed-compatible'
        or phh_corpus_governance_receipt['policy']['real_money'] is not False
        or phh_corpus_governance_receipt['policy']['permitted_use'] != ['model-training']
        or not phh_corpus_governance_receipt['temporal']['strict_order_verified']):
    raise RuntimeError('governanca PHH nao autoriza treino; operacao bloqueada.')
phh_governance_bytes = json.dumps(
    phh_corpus_governance_receipt, sort_keys=True, separators=(',', ':'),
    ensure_ascii=True,
).encode('utf-8')
phh_governance_sha256 = hashlib.sha256(phh_governance_bytes).hexdigest()
with open('phh_corpus_governance_receipt.json', 'w', encoding='utf-8') as receipt_file:
    json.dump(phh_corpus_governance_receipt, receipt_file, indent=2, sort_keys=True)

# Unica etapa de ajuste dos priors: exclusivamente registros do train.
stats = defaultdict(
    lambda: dict(hands=0, vpip=0, pfr=0, faced=0, folded=0, calls=0, raises=0)
)
train_stratum_parsed = Counter()
train_sample_hands = 0
for record in hand_records:
    if record['split'] != 'train':
        continue
    train_sample_hands += 1
    train_stratum_parsed[record['stratum']] += 1
    for player_id, delta in zip(record['hashed_player_id'], record['player_deltas']):
        player_stats = stats[player_id]
        for field, value in zip(STAT_FIELDS, delta):
            player_stats[field] += value
if train_sample_hands != split_summary['train']['hands']:
    raise RuntimeError('ajuste train-only divergiu do recibo de split')
pop = [s for s in stats.values() if s['hands'] >= MIN_HANDS_PER_PLAYER]
if not pop:
    raise RuntimeError('train insuficiente para priors; aumente SAMPLE_HANDS')


def pct(vals, q):
    vals = sorted(vals)
    return vals[min(int(q * len(vals)), len(vals) - 1)]


vpips = [s['vpip'] / s['hands'] for s in pop]
pfrs = [s['pfr'] / s['hands'] for s in pop]
f2bs = [s['folded'] / s['faced'] for s in pop if s['faced'] >= 30]
aggrs = [s['raises'] / (s['calls'] + s['raises']) for s in pop if (s['calls'] + s['raises']) >= 30]
if not (vpips and pfrs and f2bs and aggrs):
    raise RuntimeError('metricas populacionais insuficientes.')
PRIORS = {
    'source': 'PHH v3 handhq (CC BY 4.0), equal-strata; nao representativo da populacao',
    'source_commit': DATA_COMMIT,
    'source_tree': DATA_TREE,
    'sampling_seed': DATA_SAMPLE_SEED,
    'training_split': 'train',
    'split_receipt': 'phh_split_receipt.json',
    'split_receipt_sha256': split_receipt_sha256,
    'stratum_hands': dict(sorted(train_stratum_parsed.items())),
    'source_files': source_files,
    'total_ingested_hands': parsed,
    'sample_hands': train_sample_hands,
    'held_out_hands': {
        'validation': split_summary['validation']['hands'],
        'test': split_summary['test']['hands'],
    },
    'players_measured': len(pop),
    'fold_to_bet': {'p25': pct(f2bs, .25), 'median': pct(f2bs, .5), 'p75': pct(f2bs, .75)},
    'aggression': {'p25': pct(aggrs, .25), 'median': pct(aggrs, .5), 'p75': pct(aggrs, .75)},
    'vpip': {'p25': pct(vpips, .25), 'median': pct(vpips, .5), 'p75': pct(vpips, .75)},
    'pfr': {'p25': pct(pfrs, .25), 'median': pct(pfrs, .5), 'p75': pct(pfrs, .75)},
}
with open('priors.json', 'w', encoding='utf-8') as priors_file:
    json.dump(PRIORS, priors_file, indent=2, sort_keys=True)
box('PRIORS COM COBERTURA EQUAL-STRATA (NAO POPULACIONAIS)', [
    f"maos train: {train_sample_hands:,}/{parsed:,}  jogadores: {len(pop):,}",
    f"split receipt sha256: {split_receipt_sha256}",
    f"fold-to-bet mediano : {PRIORS['fold_to_bet']['median']:.2f}",
    f"agressao mediana    : {PRIORS['aggression']['median']:.2f}",
    f"VPIP mediano        : {PRIORS['vpip']['median']:.2f}   PFR: {PRIORS['pfr']['median']:.2f}",
    '',
    'Uso experimental: compare por sensibilidade contra prior neutro e por site/stake.',
    'Nao trate a mediana equal-strata como prevalencia da populacao de jogadores.',
])

# ============================================================
# [3/5] perfis humanos calibrados (fish/reg) + sondas -> POOL
# ============================================================
box('[3/5] Calibrando perfis humanos a partir da distribuicao REAL')
# limiar de forca tal que P(preflop_strength >= t) = alvo (empirico, 20k maos)
deck_rng = random.Random(7)
samples = []
for _ in range(20_000):
    d = Deck(deck_rng.randrange(1 << 30))
    d.shuffle()
    samples.append(preflop_strength(d.deal(2)))
samples.sort()


def strength_threshold(target_frac):
    idx = max(0, min(len(samples) - 1, int((1 - target_frac) * len(samples))))
    return samples[idx]


FISH = dict(vpip=PRIORS['vpip']['p75'], pfr=PRIORS['pfr']['p25'],
            f2b=PRIORS['fold_to_bet']['p25'], aggr=PRIORS['aggression']['p25'])
REG = dict(vpip=PRIORS['vpip']['p25'], pfr=PRIORS['pfr']['p75'],
           f2b=PRIORS['fold_to_bet']['p75'], aggr=PRIORS['aggression']['p75'])
for prof in (FISH, REG):
    prof['t_vpip'] = strength_threshold(prof['vpip'])
    prof['t_pfr'] = strength_threshold(prof['pfr'])
print(f"  FISH (solto-passivo): entra {FISH['vpip']:.0%}, abre {FISH['pfr']:.0%}, folda {FISH['f2b']:.0%}")
print(f"  REG  (apertado-agr.): entra {REG['vpip']:.0%}, abre {REG['pfr']:.0%}, folda {REG['f2b']:.0%}")

# indices das features (contrato do encoder v1)
I_POT, I_TOCALL, I_STRENGTH = 104, 105, 110
I_STREET0 = 111


def spec_policy(spec, rng):
    """Constroi uma OpponentPolicy (features, mask) -> acao a partir de um spec."""

    def legal(mask):
        return [i for i, ok in enumerate(mask) if ok]

    if spec == 'random':
        return lambda f, m, r=rng: r.choice(legal(m))
    if spec == 'maniac':
        def p(f, m):
            return 4 if m[4] else (1 if m[1] else legal(m)[0])
        return p
    if spec == 'station':
        def p(f, m):
            return 1 if m[1] else legal(m)[0]
        return p
    if spec == 'nit':
        def p(f, m):
            to_call = f[I_TOCALL]
            if to_call > 0.004:  # ~mais que um min-raise em fichas normalizadas
                return 0 if m[0] else (1 if m[1] else legal(m)[0])
            if m[2]:
                return 2  # pressao minima constante
            return 1 if m[1] else legal(m)[0]
        return p
    if spec == 'heur':  # o velho heuristico, via feature de forca + pot odds
        def p(f, m):
            s, pot, tc = f[I_STRENGTH], f[I_POT], f[I_TOCALL]
            if s >= 0.72 and m[3]:
                return 3
            if tc <= 0:
                return 1 if m[1] else legal(m)[0]
            po = tc / (pot + tc) if (pot + tc) > 0 else 1.0
            if s >= po and m[1]:
                return 1
            return 0 if m[0] else legal(m)[0]
        return p
    if spec in ('fish', 'reg'):
        prof = FISH if spec == 'fish' else REG
        def p(f, m, prof=prof, r=rng):
            s, pot, tc = f[I_STRENGTH], f[I_POT], f[I_TOCALL]
            preflop = f[I_STREET0] >= 0.5
            if preflop and tc > 0:
                if s >= prof['t_pfr'] and m[3]:
                    return 3
                if s >= prof['t_vpip'] and m[1]:
                    return 1
                # abaixo da faixa: fish ainda paga as vezes (e solto), reg folda
                if spec == 'fish' and m[1] and r.random() < 0.35:
                    return 1
                return 0 if m[0] else (1 if m[1] else legal(m)[0])
            if tc <= 0:
                if s >= 0.80 and r.random() < prof['aggr'] and m[3]:
                    return 3
                return 1 if m[1] else legal(m)[0]
            po = tc / (pot + tc) if (pot + tc) > 0 else 1.0
            if r.random() < prof['f2b'] and s < po + 0.15:
                return 0 if m[0] else (1 if m[1] else legal(m)[0])
            return 1 if m[1] else legal(m)[0]
        return p
    raise ValueError(spec)


POOL = ['snapshot', 'snapshot', 'random', 'heur', 'maniac', 'station', 'nit', 'fish', 'reg']

from poker_arena.bots import HeuristicBot, MonteCarloBot, RandomBot

class Maniac:
    def act(self, obs):
        if ActionType.ALL_IN in obs.legal_actions:
            return Action(ActionType.ALL_IN)
        return Action(ActionType.CALL if ActionType.CALL in obs.legal_actions else ActionType.CHECK)


class Station:
    def act(self, obs):
        if ActionType.CHECK in obs.legal_actions:
            return Action(ActionType.CHECK)
        return Action(ActionType.CALL)


class Nit:
    def act(self, obs):
        legal = obs.legal_actions
        if obs.to_call > 0:
            if obs.to_call <= 10 and ActionType.CALL in legal:
                return Action(ActionType.CALL)
            return Action(ActionType.FOLD)
        if ActionType.RAISE in legal:
            return Action(ActionType.RAISE, obs.min_raise_to)
        return Action(ActionType.CHECK)


def custom_bb100(hero_act, mk_villain, hands, seed, n=6, stack=1000, sb=10, bb=20):
    if hands <= 0 or n < 2 or stack <= 0 or not (0 < sb < bb <= stack):
        raise ValueError('configuracao de avaliacao invalida.')
    villain_rng = random.Random(seed ^ 0x5EED5EED5EED5EED)
    villains = [mk_villain(villain_rng.randrange(1 << 63)) for _ in range(n - 1)]
    rng = random.Random(seed)
    total = 0
    for h in range(hands):
        players = [Player(f'P{i}', stack) for i in range(n)]
        hand = Hand(players, button=h % n, small_blind=sb, big_blind=bb,
                    seed=rng.randrange(1 << 30))
        hand.start()
        resolved = False
        for _transition in range(400):
            while not hand.round_complete():
                seat = hand.to_act
                obs = observation_for(hand)
                a = hero_act(obs) if seat == 0 else villains[seat - 1].act(obs)
                hand.apply(a)
            live = [p for p in hand.players if p.status != PlayerStatus.FOLDED]
            if len(live) <= 1 or len(hand.board) >= 5:
                hand.resolve()
                resolved = True
                break
            hand.advance_street()
        if not resolved:
            raise RuntimeError('mao nao resolveu dentro do limite de transicoes.')
        total += players[0].stack - stack
    return total / hands / bb * 100


# ============================================================
# [4/5] treino PPO contra o POOL (mesas paralelas)
# ============================================================
class GymPokerEnv(gym.Env):
    def __init__(self, worker_index, seed):
        super().__init__()
        self._worker_index = int(worker_index)
        self._base_seed = int(seed)
        self.env = SelfPlayEnv(n_players=6, seed=self._base_seed)
        self.observation_space = spaces.Box(0.0, 5.0, (FEATURE_SIZE,), dtype=np.float32)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self._m = [True] * N_ACTIONS
        self._rng = random.Random(self._base_seed)
        self._np_rng = np.random.default_rng(self._base_seed)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        effective_seed = self._base_seed if seed is None else int(seed)
        if seed is not None:
            self._rng.seed(effective_seed)
            self._np_rng = np.random.default_rng(effective_seed)
        o, self._m = self.env.reset(seed=effective_seed if seed is not None else None)
        return np.asarray(o, np.float32), {}

    def step(self, a):
        o, self._m, r, d = self.env.step(int(a))
        return np.asarray(o, np.float32), float(r), bool(d), False, {}

    def action_masks(self):
        return np.asarray(self._m, bool)

    def set_opponent_spec(self, spec):
        if spec == 'snapshot':
            sess = ort.InferenceSession('current.onnx', providers=['CPUExecutionProvider'])
            iname = sess.get_inputs()[0].name
            def opp(feats, mask):
                logits = sess.run(None, {iname: np.asarray([feats], np.float32)})[0][0]
                idx = np.array([i for i, ok in enumerate(mask) if ok])
                z = logits[idx]
                z -= z.max()
                pr = np.exp(z)
                pr /= pr.sum()
                return int(self._np_rng.choice(idx, p=pr))
            self.env.set_opponent(opp)
        else:
            self.env.set_opponent(spec_policy(spec, self._rng))


def make_env(worker_index):
    worker_seed = TRAINING_SEED + 10_000 * (worker_index + 1)
    def initialize():
        return GymPokerEnv(worker_index, worker_seed)
    return initialize


vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)], start_method='spawn')
model = MaskablePPO('MlpPolicy', vec_env, n_steps=512, batch_size=512, ent_coef=0.005,
                    gamma=0.99, gae_lambda=0.95, policy_kwargs=dict(net_arch=[256, 256]),
                    verbose=0, device='cpu', seed=TRAINING_SEED)


def export_onnx(path):
    dev = model.device
    model.policy.to('cpu').eval()

    class _Onnx(torch.nn.Module):
        def __init__(s):
            super().__init__()
            s.fe = model.policy.features_extractor
            s.mlp = model.policy.mlp_extractor
            s.an = model.policy.action_net

        def forward(s, obs):
            f = s.fe(obs)
            lp, _ = s.mlp(f)
            return s.an(lp)

    torch.onnx.export(_Onnx(), torch.zeros(1, FEATURE_SIZE), path, input_names=['obs'],
                      output_names=['logits'],
                      dynamic_axes={'obs': {0: 'batch'}, 'logits': {0: 'batch'}},
                      opset_version=17, dynamo=False)
    model.policy.to(dev).train()


pool_rng = random.Random(TRAINING_SEED + 999_983)


def assign_pool():
    """>=2 snapshots (ancora do self-play) + maniac e nit GARANTIDOS por rotacao.

    Licao do gate v2 (medida): -217 vs maniac e -27 vs nit — sem presenca
    garantida, o agente esquece como punir o all-in maniaco e o rouba-pote."""
    specs = ['snapshot', 'snapshot', 'maniac', 'nit']
    resto = [s for s in POOL if s != 'snapshot']
    specs += pool_rng.sample(resto, k=max(0, N_ENVS - len(specs)))
    pool_rng.shuffle(specs)
    for i in range(N_ENVS):
        vec_env.env_method('set_opponent_spec', specs[i % len(specs)], indices=[i])
    return specs


class Treino(BaseCallback):
    def __init__(self):
        super().__init__()
        self._t0 = None
        self._n = 0
        self._best = -1e9

    def _on_training_start(self):
        self._t0 = time.time()
        export_onnx('current.onnx')
        specs = assign_pool()
        box('POOL INICIAL DAS MESAS', [', '.join(specs)])
        box('NAO TEM PERGUNTA IDIOTA', [
            'P: Por que treinar contra maniac/station/nit e nao so contra mim mesmo?',
            'R: Porque a analise devastadora mostrou: self-play puro deixa margens',
            '   finas contra estilos que ele nunca viu. Poker de gente tem gente.',
        ])

    def _on_step(self):
        n = self.num_timesteps
        if n // EVAL_EVERY > self._n:
            self._n = n // EVAL_EVERY
            export_onnx('current.onnx')
            sess = ort.InferenceSession('current.onnx', providers=['CPUExecutionProvider'])
            iname = sess.get_inputs()[0].name

            def hero_act(obs):
                from poker_arena.ml.encoder import legal_mask, to_action
                logits = sess.run(None, {iname: np.asarray([encode(obs)], np.float32)})[0][0]
                mask = legal_mask(obs)
                best = max((i for i in range(N_ACTIONS) if mask[i]), key=lambda i: float(logits[i]))
                return to_action(obs, best)

            bh_blocks = [eval_bb100(
                hero_act, 'heuristic', hands=CHECKPOINT_SELECTION_HANDS, seed=s,
            ) for s in CHECKPOINT_SELECTION_SEEDS]
            bman_blocks = [custom_bb100(
                hero_act, lambda _s: Maniac(), CHECKPOINT_SELECTION_HANDS, s,
            ) for s in CHECKPOINT_SELECTION_SEEDS]
            bnit_blocks = [custom_bb100(
                hero_act, lambda _s: Nit(), CHECKPOINT_SELECTION_HANDS, s,
            ) for s in CHECKPOINT_SELECTION_SEEDS]
            if not np.isfinite([*bh_blocks, *bman_blocks, *bnit_blocks]).all():
                raise RuntimeError('painel de selecao produziu metrica nao finita.')
            bh, bman, bnit = map(float, map(np.mean, (bh_blocks, bman_blocks, bnit_blocks)))
            # score composto = a MESMA regua do gate (licao do v2: selecionar so
            # por vs-heuristico escolhe checkpoints que esquecem maniac/nit)
            score = bh + bman + bnit
            sps = n / max(time.time() - self._t0, 1e-9)
            eta = (TOTAL_STEPS - n) / sps / 60 if sps > 0 else 0
            print(f'  {barra(n / TOTAL_STEPS)}  heur {bh:+.0f}  maniac {bman:+.0f}  '
                  f'nit {bnit:+.0f}  (~{eta:.0f} min restantes)')
            if score > self._best:
                self._best = score
                export_onnx('best_pool.onnx')
                best_sha256 = hashlib.sha256(Path('best_pool.onnx').read_bytes()).hexdigest()
                selection_receipt = {
                    'schema_version': 1, 'checkpoint_step': int(n),
                    'artifact_sha256': best_sha256, 'score': float(score),
                    'components_bb100': {
                        'heuristic': bh_blocks, 'maniac': bman_blocks, 'nit': bnit_blocks,
                    },
                    'selection_seed_panel': list(CHECKPOINT_SELECTION_SEEDS),
                    'hands_per_seed': CHECKPOINT_SELECTION_HANDS,
                }
                Path('best_pool_receipt.json').write_text(
                    json.dumps(selection_receipt, indent=2, sort_keys=True), encoding='utf-8',
                )
                print(f'    ^ MARCO! melhor score composto ({score:+.0f}) - salvo')
            specs = assign_pool()
            print(f'    pool rotacionado: {", ".join(specs)}')
        return True


box('[4/5] TREINANDO o Expert v2 contra a populacao', [
    f'{N_ENVS} mesas paralelas - pool: {", ".join(sorted(set(POOL)))}',
    f'{TOTAL_STEPS:,} passos - eval a cada {EVAL_EVERY:,}',
])
model.learn(total_timesteps=TOTAL_STEPS, callback=Treino())
export_onnx('expert_v2.onnx')
print('treino concluido -> expert_v2.onnx')

# ============================================================
# [5/5] GATE DEVASTADOR - v1 vs v2, pareado, com sondas
# ============================================================
box('[5/5] GATE DEVASTADOR (a mesma regua da auditoria)')





GATE = {
    'random': lambda s: RandomBot(seed=s),
    'heuristic': lambda s: HeuristicBot(seed=s),
    'montecarlo': lambda s: MonteCarloBot(seed=s, n_samples=120),
    'maniac': lambda s: Maniac(),
    'station': lambda s: Station(),
    'nit': lambda s: Nit(),
}


def battery(bot_class, onnx_path, manifest_path, seeds):
    out = {}
    for name, mk in GATE.items():
        vals = [custom_bb100(
            bot_class(onnx_path, manifest_path=manifest_path, seed=s).act,
            mk, GATE_HANDS, s,
        ) for s in seeds]
        if len(vals) != len(seeds) or not np.isfinite(vals).all():
            raise RuntimeError(f'bateria invalida para {name}.')
        out[name] = vals
    return out


def paired_bootstrap_ci(values, confidence, seed):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 1 or len(values) < 20 or not np.isfinite(values).all():
        raise RuntimeError('bootstrap exige ao menos 20 blocos finitos.')
    rng = np.random.default_rng(seed)
    means = rng.choice(values, size=(20_000, len(values)), replace=True).mean(axis=1)
    alpha = 1.0 - confidence
    return (float(values.mean()), float(np.quantile(means, alpha / 2)),
            float(np.quantile(means, 1 - alpha / 2)))


# desafiante = MELHOR checkpoint; ausencia ou receipt divergente falha fechado.
v2_source = Path('best_pool.onnx')
selection_receipt_path = Path('best_pool_receipt.json')
if not v2_source.is_file() or not selection_receipt_path.is_file():
    raise RuntimeError('checkpoint selecionado e receipt sao obrigatorios; sem fallback final.')
try:
    selection_receipt = json.loads(selection_receipt_path.read_text(encoding='utf-8'))
except (OSError, UnicodeDecodeError, json.JSONDecodeError) as exc:
    raise RuntimeError('receipt de selecao invalido.') from exc
selected_sha256 = hashlib.sha256(v2_source.read_bytes()).hexdigest()
if (selection_receipt.get('schema_version') != 1
        or selection_receipt.get('artifact_sha256') != selected_sha256
        or selection_receipt.get('selection_seed_panel') != list(CHECKPOINT_SELECTION_SEEDS)
        or selection_receipt.get('hands_per_seed') != CHECKPOINT_SELECTION_HANDS
        or type(selection_receipt.get('checkpoint_step')) is not int
        or selection_receipt['checkpoint_step'] <= 0
        or not math.isfinite(float(selection_receipt.get('score', math.nan)))):
    raise RuntimeError('receipt de selecao nao vincula o melhor checkpoint atual.')
candidate_dir = Path('candidate_bundle')
if candidate_dir.exists():
    shutil.rmtree(candidate_dir)
candidate_dir.mkdir()
V2_PATH = candidate_dir / 'poker_expert.onnx'
shutil.copyfile(v2_source, V2_PATH)
candidate_selection_receipt = candidate_dir / 'best_pool_receipt.json'
shutil.copyfile(selection_receipt_path, candidate_selection_receipt)
selection_receipt_sha256 = hashlib.sha256(
    candidate_selection_receipt.read_bytes(),
).hexdigest()
v2_sha256 = hashlib.sha256(V2_PATH.read_bytes()).hexdigest()
candidate_manifest = {
    'schema_version': 1,
    'artifacts': [{
        'path': V2_PATH.name, 'state': 'candidate', 'installed': True,
        'sha256': v2_sha256,
        'governance': {
            'license': {
                'status': 'verified', 'id': CANDIDATE_LICENSE_ID,
                'reference': CANDIDATE_LICENSE_REFERENCE,
            },
            'lineage': {
                'status': 'verified', 'id': CANDIDATE_LINEAGE_ID,
                'reference': CANDIDATE_LINEAGE_REFERENCE,
            },
        },
        'inputs': [{'name': 'obs', 'dtype': 'float32', 'shape': ['batch', FEATURE_SIZE]}],
        'outputs': [{'name': 'logits', 'dtype': 'float32', 'shape': ['batch', N_ACTIONS]}],
        'training_context': {
            'project_commit': PROJECT_COMMIT, 'dataset_commit': DATA_COMMIT,
            'dataset_tree': DATA_TREE,
            'dataset_split_receipt_sha256': split_receipt_sha256,
            'phh_corpus_governance_receipt_sha256': phh_governance_sha256,
        },
    }],
}
V2_MANIFEST_PATH = candidate_dir / 'MANIFEST.json'
V2_MANIFEST_PATH.write_text(
    json.dumps(candidate_manifest, indent=2, sort_keys=True), encoding='utf-8',
)
candidate_artifact = verify_evaluation_candidate(
    V2_PATH, 'expert', manifest_path=V2_MANIFEST_PATH,
)
if candidate_artifact.usage != 'evaluation':
    raise RuntimeError('candidato nao recebeu autorizacao exclusiva de avaliacao.')
print(f'  desafiante governado: {V2_PATH} ({candidate_artifact.sha256})')

print('  medindo v1 (o campeao atual)...')
v1 = battery(MLBot, V1_PATH, V1_MANIFEST_PATH, GATE_SEEDS)
print('  medindo estabilidade do baseline em painel de seeds independente...')
v1_control_independent = battery(
    MLBot, V1_PATH, V1_MANIFEST_PATH, CONTROL_SEEDS,
)
print('  medindo v2 (o desafiante)...')
v2 = battery(EvaluationMLBot, V2_PATH, V2_MANIFEST_PATH, GATE_SEEDS)

deltas = {k: np.asarray(v2[k]) - np.asarray(v1[k]) for k in GATE}
control_blocks = np.vstack([
    np.asarray(v1_control_independent[name]) - np.asarray(v1[name]) for name in GATE
]).mean(axis=0)
family_size = len(GATE) + 2  # seis confrontos + agregado + controle
simultaneous_confidence = 1.0 - 0.05 / family_size  # Bonferroni, FWER 95%
control_mean, control_low, control_high = paired_bootstrap_ci(
    control_blocks, simultaneous_confidence, 20260798,
)
summary = {}
linhas = []
for index, name in enumerate(GATE):
    mean, low, high = paired_bootstrap_ci(
        deltas[name], simultaneous_confidence, 20260717 + index,
    )
    summary[name] = {'mean_delta_bb100': mean, 'simultaneous_ci': [low, high]}
    linhas.append(f'{name:12} delta {mean:+7.1f}  IC simultaneo [{low:+7.1f}, {high:+7.1f}]')
aggregate_blocks = np.vstack([deltas[name] for name in GATE]).mean(axis=0)
agg_mean, agg_low, agg_high = paired_bootstrap_ci(
    aggregate_blocks, simultaneous_confidence, 20260799,
)
linhas += [
    '',
    f'AGREGADO     delta {agg_mean:+7.1f}  IC simultaneo [{agg_low:+7.1f}, {agg_high:+7.1f}]',
    (f'ESTABILIDADE delta {control_mean:+7.1f}  IC simultaneo '
     f'[{control_low:+7.1f}, {control_high:+7.1f}]'),
]
box('GATE v2-v1 pareado + estabilidade baseline em painel independente', linhas)

preregistered_design = (
    bool(GATE_POWER_ANALYSIS_ID)
    and math.isfinite(GATE_MDE_BB100) and GATE_MDE_BB100 > 0.0
    and math.isfinite(GATE_NONINFERIORITY_MARGIN_BB100)
    and GATE_NONINFERIORITY_MARGIN_BB100 >= 0.0
    and math.isfinite(GATE_CONTROL_EQUIVALENCE_MARGIN_BB100)
    and GATE_CONTROL_EQUIVALENCE_MARGIN_BB100 >= 0.0
)
interval_half_widths = [
    (summary[name]['simultaneous_ci'][1] - summary[name]['simultaneous_ci'][0]) / 2.0
    for name in GATE
] + [
    (agg_high - agg_low) / 2.0,
    (control_high - control_low) / 2.0,
]
checks = {
    'preregistered_design': preregistered_design,
    'aggregate_superiority': agg_low > 0.0,
    'all_matchups_noninferior': all(
        summary[name]['simultaneous_ci'][0] > -GATE_NONINFERIORITY_MARGIN_BB100
        for name in GATE
    ) if preregistered_design else False,
    'independent_blocks': (
        len(GATE_SEEDS) >= GATE_REQUIRED_BLOCKS
        and len(CONTROL_SEEDS) >= GATE_REQUIRED_BLOCKS
        and set(GATE_SEEDS).isdisjoint(CONTROL_SEEDS)
    ),
    'baseline_seed_panel_stable': (
        control_low >= -GATE_CONTROL_EQUIVALENCE_MARGIN_BB100
        and control_high <= GATE_CONTROL_EQUIVALENCE_MARGIN_BB100
    ) if preregistered_design else False,
    'precision_supports_mde': (
        all(width <= GATE_MDE_BB100 for width in interval_half_widths)
        if preregistered_design else False
    ),
}
passa = all(checks.values())
v1_sha256 = hashlib.sha256(Path(V1_PATH).read_bytes()).hexdigest()
v2_sha256 = hashlib.sha256(Path(V2_PATH).read_bytes()).hexdigest()
gate_report = {
    'schema_version': 3, 'passed': passa, 'checks': checks,
    'project_commit': PROJECT_COMMIT, 'baseline_revision': HF_BASELINE_REVISION,
    'baseline_sha256': v1_sha256, 'candidate_sha256': v2_sha256,
    'baseline_manifest_sha256': baseline_artifact.manifest_sha256,
    'candidate_manifest_sha256': candidate_artifact.manifest_sha256,
    'dataset_commit': DATA_COMMIT, 'dataset_tree': DATA_TREE,
    'dataset_split_receipt_sha256': split_receipt_sha256,
    'phh_corpus_governance_receipt_sha256': phh_governance_sha256,
    'checkpoint_selection_receipt_sha256': selection_receipt_sha256,
    'gate_hands_per_block': GATE_HANDS, 'gate_seeds': list(GATE_SEEDS),
    'control_seeds': list(CONTROL_SEEDS),
    'block_unit': 'independent_seed_with_paired_duplicate_deals',
    'required_blocks': GATE_REQUIRED_BLOCKS,
    'multiple_testing': {
        'method': 'bonferroni', 'familywise_alpha': 0.05,
        'family_size': family_size,
        'family_members': [*GATE.keys(), 'aggregate', 'stability_control'],
    },
    'power_analysis_id': GATE_POWER_ANALYSIS_ID or None,
    'minimum_detectable_effect_bb100': (
        GATE_MDE_BB100 if math.isfinite(GATE_MDE_BB100) else None
    ),
    'noninferiority_margin_bb100': (
        GATE_NONINFERIORITY_MARGIN_BB100
        if math.isfinite(GATE_NONINFERIORITY_MARGIN_BB100) else None
    ),
    'control_equivalence_margin_bb100': (
        GATE_CONTROL_EQUIVALENCE_MARGIN_BB100
        if math.isfinite(GATE_CONTROL_EQUIVALENCE_MARGIN_BB100) else None
    ),
    'stability_control': {
        'definition': (
            'baseline independent seed panel minus gate seed panel; non-degenerate '
            'behavioral stability control for evaluator and seed-panel drift'
        ),
        'mean_delta_bb100': control_mean,
        'simultaneous_ci': [control_low, control_high],
    },
    'aggregate': {
        'mean_delta_bb100': agg_mean, 'simultaneous_ci': [agg_low, agg_high],
    },
    'matchups': summary,
}
with open('gate_report.json', 'w', encoding='utf-8') as report_file:
    json.dump(gate_report, report_file, indent=2, sort_keys=True)

if PUBLISH_ARTIFACTS:
    publish_token = userdata.get('HF_TOKEN')
    if not isinstance(publish_token, str) or not publish_token:
        raise RuntimeError('HF_TOKEN e obrigatorio somente no instante de publicar.')
    api = HfApi(token=publish_token)
    publish_token = ''
    api.create_repo(HF_REPO_ID, repo_type='model', exist_ok=True, private=True)
    candidate_root = f'candidates/{v2_sha256[:16]}'
    artifact_commit = api.upload_file(
        path_or_fileobj=V2_PATH, path_in_repo=f'{candidate_root}/poker_expert.onnx',
        repo_id=HF_REPO_ID, repo_type='model',
    )
    manifest_commit = api.upload_file(
        path_or_fileobj=V2_MANIFEST_PATH, path_in_repo=f'{candidate_root}/MANIFEST.json',
        repo_id=HF_REPO_ID, repo_type='model',
    )
    report_commit = api.upload_file(
        path_or_fileobj='gate_report.json', path_in_repo=f'{candidate_root}/gate_report.json',
        repo_id=HF_REPO_ID, repo_type='model',
    )
    split_receipt_commit = api.upload_file(
        path_or_fileobj='phh_split_receipt.json',
        path_in_repo=f'{candidate_root}/phh_split_receipt.json',
        repo_id=HF_REPO_ID, repo_type='model',
    )
    governance_receipt_commit = api.upload_file(
        path_or_fileobj='phh_corpus_governance_receipt.json',
        path_in_repo=f'{candidate_root}/phh_corpus_governance_receipt.json',
        repo_id=HF_REPO_ID, repo_type='model',
    )
    selection_receipt_commit = api.upload_file(
        path_or_fileobj=candidate_selection_receipt,
        path_in_repo=f'{candidate_root}/best_pool_receipt.json',
        repo_id=HF_REPO_ID, repo_type='model',
    )
    receipt = {
        'candidate_root': candidate_root, 'artifact_commit': artifact_commit.oid,
        'manifest_commit': manifest_commit.oid,
        'report_commit': report_commit.oid,
        'split_receipt_commit': split_receipt_commit.oid,
        'split_receipt_sha256': split_receipt_sha256,
        'governance_receipt_commit': governance_receipt_commit.oid,
        'governance_receipt_sha256': phh_governance_sha256,
        'selection_receipt_commit': selection_receipt_commit.oid,
        'selection_receipt_sha256': selection_receipt_sha256,
        'canonical_updated': False,
    }
    with open('hf_publish_receipt.json', 'w', encoding='utf-8') as receipt_file:
        json.dump(receipt, receipt_file, indent=2, sort_keys=True)
    if passa:
        box('CANDIDATO APROVADO PELOS GATES INTERNOS', [
            f'candidate sha256: {v2_sha256}',
            'publicado somente no caminho imutavel; canonico inalterado.',
            'promocao exige revisao independente do recibo e do gate_report.',
        ])
    else:
        box('NAO PROMOVIDO', ['candidato arquivado; um ou mais gates falharam.'])
    del api
else:
    box('CHECK-ONLY', [
        f'gate passou: {passa}; candidato sha256: {v2_sha256}',
        'nenhum artefato foi publicado (POKER_PUBLISH_ARTIFACTS != 1).',
    ])
print('\nFIM. Evidencias locais: priors.json, phh_split_receipt.json, '
      'phh_corpus_governance_receipt.json e gate_report.json')
